# 05. Análise: Respondendo às Perguntas de Negócio e Discussão Geral
Meta: **R$ 2.000.000 em novembro/2018**. Referência: Black Friday 2017.
Período de análise: jan/2017 a ago/2018 (as pontas da base têm poucos pedidos).


## Resumo executivo
**Pergunta:** terei estoque para vender R$ 2M em novembro (Black Friday)?

**Resposta curta:** o efeito natural da Black Friday sobre a base de 2018 projeta **R$ 1,50M (75% da meta)**. Atingir R$ 2M exigiria o **dobro das unidades** vendidas em nov/2017 na curva A. A recomendação é estocar para o cenário projetado (~1,5x) e condicionar a compra adicional a ações comerciais que fechem o gap de R$ 504 mil.

*Detalhamento por pergunta abaixo; discussão completa ao final.*

## P1: Quais produtos formam as curvas A, B e C?
A classificação é feita por receita acumulada: **A** = até 80% da receita, **B** = até 95%, **C** = restante. A classificação fica salva na view `vw_curva_abc`, reaproveitada na P5.

In [0]:
CREATE OR REPLACE VIEW mvp_olist.gold.vw_curva_abc
COMMENT 'Curva ABC de produtos por receita (jan/2017 a ago/2018): A = 80%, B = 15%, C = 5%'
AS
WITH receita AS (
  SELECT p.product_id, p.categoria_pt,
         SUM(f.valor_venda) AS receita,
         SUM(f.quantidade)  AS unidades
  FROM mvp_olist.gold.fato_vendas f
  JOIN mvp_olist.gold.dim_produto p ON f.sk_produto = p.sk_produto
  JOIN mvp_olist.gold.dim_tempo t   ON f.sk_data = t.sk_data
  WHERE t.data BETWEEN DATE'2017-01-01' AND DATE'2018-08-31'
  GROUP BY p.product_id, p.categoria_pt
),
acumulado AS (
  SELECT *,
         SUM(receita) OVER (ORDER BY receita DESC, product_id ROWS UNBOUNDED PRECEDING)
           / SUM(receita) OVER () AS pct_acumulado
  FROM receita
)
SELECT *,
       CASE WHEN pct_acumulado <= 0.80 THEN 'A'
            WHEN pct_acumulado <= 0.95 THEN 'B'
            ELSE 'C' END AS curva
FROM acumulado;

In [0]:
SELECT curva,
       COUNT(*)                                                 AS produtos,
       ROUND(100 * COUNT(*) / SUM(COUNT(*)) OVER (), 1)         AS pct_produtos,
       ROUND(SUM(receita), 2)                                   AS receita,
       ROUND(100 * SUM(receita) / SUM(SUM(receita)) OVER (), 1) AS pct_receita
FROM mvp_olist.gold.vw_curva_abc
GROUP BY curva
ORDER BY curva;

curva,produtos,pct_produtos,receita,pct_receita
A,8429,25.9,10759397.07,80.0
B,11180,34.3,2017583.90,15.0
C,12969,39.8,672548.71,5.0


**Discussão P1:** 8.429 produtos (25,9% do portfólio) são responsáveis por 80% da receita (curva A). A concentração é menor que o "20/80" clássico, típico de marketplace com cauda longa. Para o planejamento de estoque, a curva A é o grupo onde a ruptura apresenta maior risco de impacto negativo às vendas e deve ter prioridade de compra; a curva C (39,8% dos produtos, 5% da receita) pode operar com estoque mínimo ou sob demanda.

## P2: Qual a participação de cada categoria na Black Friday 2017?
Janela da Black Friday: sexta a segunda (24/11 a 27/11/2017), definida na `dim_tempo`.

In [0]:
SELECT p.categoria_pt,
       SUM(f.quantidade)                                                    AS unidades,
       ROUND(SUM(f.valor_venda), 2)                                         AS receita,
       ROUND(100 * SUM(f.valor_venda) / SUM(SUM(f.valor_venda)) OVER (), 1) AS pct_receita_bf
FROM mvp_olist.gold.fato_vendas f
JOIN mvp_olist.gold.dim_produto p ON f.sk_produto = p.sk_produto
JOIN mvp_olist.gold.dim_tempo t   ON f.sk_data = t.sk_data
WHERE t.campanha = 'Black Friday' AND t.ano = 2017
GROUP BY p.categoria_pt
ORDER BY receita DESC
LIMIT 10;

categoria_pt,unidades,receita,pct_receita_bf
cama_mesa_banho,397,35718.47,11.7
relogios_presentes,116,25716.73,8.4
informatica_acessorios,169,24400.61,8.0
beleza_saude,190,24384.70,8.0
moveis_decoracao,295,23460.08,7.7
brinquedos,178,18771.87,6.1
esporte_lazer,178,17972.59,5.9
ferramentas_jardim,238,14814.48,4.8
perfumaria,114,12991.16,4.2
cool_stuff,72,11028.92,3.6


**Discussão P2:** nos 4 dias da Black Friday 2017, as 5 maiores categorias concentraram 43,8% da receita, lideradas por cama_mesa_banho (11,7%). É nessas categorias que um erro de previsão de estoque tem maior impacto negativo no resultado das vendas da data.

## P3: O mix de uma campanha comparável (Dia das Mães/2018) se parece com o da Black Friday?
Janela do Dia das Mães: 01/05 a 13/05/2018 (duas semanas antes da data).

In [0]:
WITH mix AS (
  SELECT t.campanha, p.categoria_pt,
         100 * SUM(f.valor_venda) / SUM(SUM(f.valor_venda)) OVER (PARTITION BY t.campanha) AS pct
  FROM mvp_olist.gold.fato_vendas f
  JOIN mvp_olist.gold.dim_produto p ON f.sk_produto = p.sk_produto
  JOIN mvp_olist.gold.dim_tempo t   ON f.sk_data = t.sk_data
  WHERE (t.campanha = 'Black Friday' AND t.ano = 2017)
     OR (t.campanha = 'Dia das Mães' AND t.ano = 2018)
  GROUP BY t.campanha, p.categoria_pt
)
SELECT categoria_pt,
       ROUND(MAX(CASE WHEN campanha = 'Black Friday' THEN pct END), 1) AS pct_black_friday_2017,
       ROUND(MAX(CASE WHEN campanha = 'Dia das Mães' THEN pct END), 1) AS pct_dia_das_maes_2018
FROM mix
GROUP BY categoria_pt
ORDER BY pct_black_friday_2017 DESC NULLS LAST
LIMIT 10;

categoria_pt,pct_black_friday_2017,pct_dia_das_maes_2018
cama_mesa_banho,11.7,5.8
relogios_presentes,8.4,14.8
informatica_acessorios,8.0,5.0
beleza_saude,8.0,9.7
moveis_decoracao,7.7,5.3
brinquedos,6.1,3.1
esporte_lazer,5.9,6.3
ferramentas_jardim,4.8,3.5
perfumaria,4.2,2.5
cool_stuff,3.6,3.3


**Discussão P3:** o mix das duas campanhas é bem diferente: relógios e presentes saltam de 8,4% para 14,8% no Dia das Mães, enquanto cama_mesa_banho e brinquedos caem pela metade. Conclui-se que o Dia das Mães não serve como referência para o estoque da Black Friday; o planejamento deve partir da BF do ano anterior.

## P4. Quanto a Black Friday representa sobre um mês normal, e qual crescimento é necessário para R$ 2M?
Mês normal = média de ago a out/2017 (três meses antes da BF).

In [0]:
WITH mensal AS (
  SELECT t.ano_mes, SUM(f.valor_venda) AS receita
  FROM mvp_olist.gold.fato_vendas f
  JOIN mvp_olist.gold.dim_tempo t ON f.sk_data = t.sk_data
  GROUP BY t.ano_mes
)
SELECT
  ROUND(MAX(CASE WHEN ano_mes = '2017-11' THEN receita END), 2)                        AS receita_nov_2017,
  ROUND(AVG(CASE WHEN ano_mes BETWEEN '2017-08' AND '2017-10' THEN receita END), 2)    AS media_ago_out_2017,
  ROUND(MAX(CASE WHEN ano_mes = '2017-11' THEN receita END)
      / AVG(CASE WHEN ano_mes BETWEEN '2017-08' AND '2017-10' THEN receita END), 2)    AS fator_sazonal_bf,
  ROUND(100 * (2000000 / MAX(CASE WHEN ano_mes = '2017-11' THEN receita END) - 1), 1) AS pct_crescimento_necessario
FROM mensal;

receita_nov_2017,media_ago_out_2017,fator_sazonal_bf,pct_crescimento_necessario
1003862.14,616613.77,1.63,99.2


**Discussão P4:** a Black Friday faz novembro render 1,63x a média dos três meses anteriores. Para atingir R$ 2M seria necessário quase dobrar (+99,2%) a receita de nov/2017.

### P4.2: Projeção para nov/2018
Aplica o fator da Black Friday (1,63) sobre a média mensal de 2018 para estimar quanto da meta vem do efeito natural da data.

In [0]:
WITH mensal AS (
  SELECT t.ano_mes, SUM(f.valor_venda) AS receita
  FROM mvp_olist.gold.fato_vendas f
  JOIN mvp_olist.gold.dim_tempo t ON f.sk_data = t.sk_data
  GROUP BY t.ano_mes
)
SELECT
  ROUND(AVG(CASE WHEN ano_mes BETWEEN '2018-01' AND '2018-08' THEN receita END), 2)                  AS media_mensal_2018,
  ROUND(1.63 * AVG(CASE WHEN ano_mes BETWEEN '2018-01' AND '2018-08' THEN receita END), 2)           AS projecao_nov_2018,
  ROUND(2000000 - 1.63 * AVG(CASE WHEN ano_mes BETWEEN '2018-01' AND '2018-08' THEN receita END), 2) AS gap_para_meta
FROM mensal;

media_mensal_2018,projecao_nov_2018,gap_para_meta
917629.68,1495736.37,504263.63


**Discussão P4 (projeção):** aplicando o fator 1,63 sobre a média mensal de 2018 (R$ 917.630), a projeção para nov/2018 é de R$ 1,50M. O efeito natural da data entrega ~75% da meta; os R$ 504 mil restantes (25%) dependem de ações comerciais adicionais.

## P5. Quantas unidades da curva A são necessárias para bater R$ 2M?
Premissa: manter o mix e o preço médio de nov/2017. Unidades necessárias = unidades de nov/2017 × (R$ 2M ÷ receita de nov/2017). O cálculo por produto fica salvo na view `vw_necessidade_estoque`.

In [0]:
CREATE OR REPLACE VIEW mvp_olist.gold.vw_necessidade_estoque
COMMENT 'Unidades necessárias por produto para atingir R$ 2M em nov/2018, mantendo o mix e o preço médio de nov/2017'
AS
WITH nov17 AS (
  SELECT p.product_id, p.categoria_pt,
         SUM(f.quantidade)  AS unidades_nov17,
         SUM(f.valor_venda) AS receita_nov17
  FROM mvp_olist.gold.fato_vendas f
  JOIN mvp_olist.gold.dim_produto p ON f.sk_produto = p.sk_produto
  JOIN mvp_olist.gold.dim_tempo t   ON f.sk_data = t.sk_data
  WHERE t.ano_mes = '2017-11'
  GROUP BY p.product_id, p.categoria_pt
)
SELECT n.*,
       a.curva,
       2000000 / SUM(n.receita_nov17) OVER ()                          AS fator_meta,
       CEIL(n.unidades_nov17 * 2000000 / SUM(n.receita_nov17) OVER ()) AS unidades_necessarias
FROM nov17 n
LEFT JOIN mvp_olist.gold.vw_curva_abc a ON n.product_id = a.product_id;

In [0]:
SELECT categoria_pt,
       COUNT(*)                  AS produtos_curva_a,
       SUM(unidades_nov17)       AS unidades_vendidas_nov17,
       SUM(unidades_necessarias) AS unidades_necessarias_nov18
FROM mvp_olist.gold.vw_necessidade_estoque
WHERE curva = 'A'
GROUP BY categoria_pt
ORDER BY unidades_necessarias_nov18 DESC
LIMIT 10;

categoria_pt,produtos_curva_a,unidades_vendidas_nov17,unidades_necessarias_nov18
cama_mesa_banho,237,654,1308
ferramentas_jardim,41,501,1002
moveis_decoracao,140,468,936
beleza_saude,128,404,808
informatica_acessorios,136,366,732
relogios_presentes,148,365,730
esporte_lazer,184,360,720
brinquedos,110,328,656
cool_stuff,91,229,458
perfumaria,62,199,398


**Discussão P5:** mantendo o mix e o preço médio de nov/2017, a meta exige o dobro das unidades vendidas na curva A. Categorias concentradas em poucos produtos merecem atenção: ferramentas_jardim precisa de 1.002 unidades em apenas 41 produtos (~24 por produto), enquanto cama_mesa_banho distribui 1.308 unidades em 237 produtos. Quanto maior a concentração, maior o risco de ruptura.

## P6. O estoque disponível cobre essa necessidade?
**Não respondida.** A base Olist não possui dados de estoque, então não é possível comparar a necessidade calculada na P5 com o estoque disponível. A pergunta foi mantida no objetivo, como pedem as instruções do MVP, e é discutida na autoavaliação e nos trabalhos futuros (ex.: aplicar o mesmo pipeline a dados reais de uma empresa que tenha estoque).

## Discussão geral
**Pergunta central:** terei estoque para vender R$ 2M em novembro (Black Friday)?

1. **Onde focar (P1):** 26% dos produtos (curva A) geram 80% da receita; é nesse grupo que a ruptura de estoque mais custa em impacto de vendas.
2. **O que vende na BF (P2/P3):** o mix da Black Friday é próprio (casa, eletrônicos, beleza) e difere do Dia das Mães (campanha com foco em presentes), então o planejamento deve partir da BF anterior, não de outra campanha.
3. **Quanto a meta exige (P4):** a BF multiplica o mês por 1,63x. Aplicado à média de 2018, projeta R$ 1,50M; a meta de R$ 2M exige R$ 504 mil (25%) além do efeito natural da data.
4. **Quanto estocar (P5):** para a meta, seria preciso o dobro das unidades de nov/2017 na curva A. Categorias concentradas em poucos produtos (ex.: ferramentas_jardim, ~24 un./produto) têm maior risco de ruptura e devem ser priorizadas na compra.
5. **O que não foi respondido (P6):** a base não possui estoque, então não é possível confirmar se o estoque disponível cobre a necessidade calculada.

**Recomendação:** para atingir a meta, estocar a curva A para o cenário projetado (~1,5x nov/2017) e condicionar a compra adicional (até 2x) a ações comerciais (conforme capacidade operacional da empresa) que justifiquem o gap, evitando capital parado em estoque.